In [17]:
import numpy as np
import torch
import torch.nn as nn

Comprobar que detecte GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## Embedding Posicional

In [19]:
class EmbeddingPosicional(nn.Module):
    def __init__(self, d_embedding, vocab_size, seq_len):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1, seq_len, d_embedding) * 0.02) # Inicialización pequeña
        self.embedding = nn.Embedding(vocab_size, d_embedding)
    
    def forward(self, x):
        out = self.embedding(x) # devuelve (b, n, d)
        _, n, _ = out.shape
        out = out + self.weights[:,:n,:]
        return out

## Capa Feedforward

In [20]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.l1 = nn.Linear(d_model, d_model*4)
        self.silu = nn.SiLU()
        self.l2 = nn.Linear(d_model*4, d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm(x)
        x = self.l1(x)
        x = self.silu(x)
        x = self.l2(x)
        x = self.dropout(x)
        return x

## Capa de atención

In [21]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_head, d_head, use_mask=False):
        super().__init__()
        self.qw = nn.Linear(d_model, d_head*n_head)
        self.kw = nn.Linear(d_model, d_head*n_head)
        self.vw = nn.Linear(d_model, d_head*n_head)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
        self.out = nn.Linear(d_head*n_head, d_model)

        self.n_head = n_head
        self.d_head = d_head
        self.scale = d_head ** 0.5
        self.use_mask = use_mask
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        b, n, d = x.shape
        x = self.norm(x)

        q = self.qw(x)
        k = self.kw(x)
        v = self.vw(x)

        q = q.reshape(b, -1, self.n_head, self.d_head)
        k = k.reshape(b, -1, self.n_head, self.d_head)
        v = v.reshape(b, -1, self.n_head, self.d_head)

        #print(q.shape)
        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale
        
        if self.use_mask:
            _, _, _, n = scores.shape
            mask = torch.tril(torch.ones(1, 1, n, n))   # (n, n)

            scores = scores.masked_fill(mask[:, :, :n, :n] == 0, float('-inf'))

        softmax_scores = scores.softmax(dim=-1)
        att = self.dropout(softmax_scores)
        out = torch.einsum('bhij,bjhd->bihd', att, v)
        #print(out.shape)
        out = self.dropout(out).reshape(b, -1, self.d_head*self.n_head)
        #print(out.shape)
        out = self.out(out)
        return out, att

## Encoder

In [22]:
class Encoder(nn.Module):
    def __init__(self, d_model=16, vocab_size=20, seq_len=32, nb_layers=5, n_head=2, d_head=8):
        super().__init__()
        self.embedding = EmbeddingPosicional(d_model, vocab_size, seq_len)
        self.ff = nn.ModuleList([FeedForward(d_model) for _ in range(nb_layers)])
        self.atts = nn.ModuleList([SelfAttention(d_model, n_head, d_head) for _ in range(nb_layers)])
        self.norm_final = nn.LayerNorm(d_model)
    
    def forward(self, x):
        x = self.embedding(x)
        for ff_layer, att_layer in zip(self.ff, self.atts):
            att_out, _ = att_layer(x)
            x = x + att_out
            ff_out = ff_layer(x)
            x = x + ff_out
        x = self.norm_final(x)
        return x

In [23]:
encoder = Encoder()
x = torch.randint(0, 20, (2, 32))
out = encoder(x)
print(out.shape)

torch.Size([2, 32, 16])


## Tokenizador

En este tokenizador se añade el token CLS

In [ ]:
import re
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

class SimpleTokenizer:
    def __init__(self, text_corpus):
        tokens = word_tokenize(text_corpus.lower())
        
        self.vocab = sorted(list(set([t for t in tokens if t.isalnum()])))
        
        self.vocab_size = len(self.vocab) + 4 
        self.word2idx = {w: i+4 for i, w in enumerate(self.vocab)}
        self.idx2word = {i+4: w for i, w in enumerate(self.vocab)}
        
        self.PAD_TOKEN = 0
        self.MASK_TOKEN = 1
        self.UNK_TOKEN = 2
        self.CLS_TOKEN = 3
        self.word2idx['[PAD]'] = 0
        self.word2idx['[MASK]'] = 1
        self.word2idx['[UNK]'] = 2
        self.word2idx['[CLS]'] = 3 # Añadimos aquí el token [CLS]

    def encode(self, text, max_len):
        tokens = word_tokenize(text.lower())
        indices = [self.word2idx.get(t, self.UNK_TOKEN) for t in tokens if t.isalnum()]
        indices = [self.CLS_TOKEN] + indices  # Añadir token [CLS] al inicio
        
        if len(indices) < max_len:
            indices += [self.PAD_TOKEN] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        return torch.tensor(indices, dtype=torch.long)
    

## Clase para el Dataset

In [25]:
class BookDataset(Dataset):
    def __init__(self, pages, tokenizer, seq_len):
        self.pages = pages
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.pages)

    def __getitem__(self, idx):
        return self.tokenizer.encode(self.pages[idx], self.seq_len)

## Modelo Final

In [ ]:
class EncoderForMaskedLM(nn.Module):
    def __init__(self, encoder, d_model, vocab_size):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: (batch, seq_len)
        embeddings = self.encoder(x) # (batch, seq_len, d_model)
        prediction_scores = self.head(embeddings) # (batch, seq_len, vocab_size)
        return prediction_scores, embeddings

Función para entrenar el modelo

In [ ]:
from tqdm import tqdm 
def train_model(pages, d_model=256, seq_len=20, epochs=100):
    full_text = " ".join(pages)
    tokenizer = SimpleTokenizer(full_text)
    dataset = BookDataset(pages, tokenizer, seq_len)
    
    loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)
    
    print(f"Vocab Size: {tokenizer.vocab_size}")

    base_encoder = Encoder(d_model=d_model, vocab_size=tokenizer.vocab_size, 
                           seq_len=seq_len, nb_layers=6, n_head=8, d_head=d_model//8)
    
    model = EncoderForMaskedLM(base_encoder, d_model, tokenizer.vocab_size)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)

    model.train()
    
    for epoch in tqdm(range(epochs)):
        total_loss = 0
        for batch in loader:
            inputs = batch.to(device) 
            labels = batch.clone().to(device) 
            
            rand = torch.rand(inputs.shape, device=device)
            mask_arr = (rand < 0.15) & (inputs != tokenizer.PAD_TOKEN) # Máscara solo en tokens no PAD
            
            if not mask_arr.any():
                continue

            labels[~mask_arr] = -100 # Ignoramos los tokens que no se han enmascarado
            inputs[mask_arr] = tokenizer.MASK_TOKEN
            
            outputs, _ = model(inputs) 
            
            loss = criterion(outputs.view(-1, tokenizer.vocab_size), labels.view(-1)) # loss de los tokens enmascarados
            
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Evita la explosión de gradientes
            
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(loader)
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    torch.save(model.state_dict(), "encoder_mlm_cls.pth")

    return model.encoder, tokenizer

Función para obtener los embeddings a partir del token CLS

In [ ]:
def get_page_embeddings(encoder, tokenizer, pages, seq_len):
    encoder.eval()
    embeddings = []
    
    with torch.no_grad():
        for page in pages:
            input_tensor = tokenizer.encode(page, seq_len).unsqueeze(0).to(device)
            
            page_emb_seq = encoder(input_tensor)
            
            cls_embedding = page_emb_seq[:, 0, :].squeeze(0).cpu().numpy() # Extraemos el embedding del token [CLS]
            embeddings.append(cls_embedding)
            
    return embeddings

## Leer ficheros pdfs

In [ ]:
import glob
import pypdf
import re

seq_len = 256
overlap = 10  # Solapamiento de palabras para mantener contexto entre chunks

def clean_page_text(text):
    """
    Limpia el texto crudo extraído del PDF antes de dividirlo.
    """
    if not text: return ""
    
    # Busca: letra + guión + posibles espacios/saltos + letra
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # elimina urls y correos
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    
    # reemplaza múltiples espacios y saltos de línea por un solo espacio
    text = re.sub(r'\s+', ' ', text).strip()
    
    # elimina caracteres muy extraños
    text = re.sub(r'[^\w\s.,!?áéíóúÁÉÍÓÚñÑüÜ]', '', text) 

    tokens = word_tokenize(text.lower())
    
    # Nos quedamos solo con tokens alfanuméricos
    clean_tokens = [t for t in tokens if t.isalnum()]
    
    return clean_tokens
    
# Cargar páginas
pdf_files = glob.glob("/home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/*.pdf")

chunks_pdfs = {}

total_pages = 0

for pdf_file in pdf_files:
    try:
        reader = pypdf.PdfReader(pdf_file)
        file_chunks = []
        
        for page in tqdm(reader.pages):
            raw_text = page.extract_text()
            
            # Limpieza inicial
            clean_text = clean_page_text(raw_text)
            
            if not clean_text:
                continue
                
            words = clean_text
            
            # Si la página tiene muy poco contenido la ingoramos
            if len(words) < 10:
                continue
                
            step = seq_len - overlap
            if step < 1: step = 1
            
            for i in range(0, len(words), step):
                # Extraemos el chunk
                chunk_words = words[i : i + seq_len]

                # Al leer el chunk, se solía quedar con "página X" al final, lo eliminamos
                if len(chunk_words) >= 2 and chunk_words[-2] == "página" and chunk_words[-1].isdigit():
                    chunk_words = chunk_words[:-2]
                # Reconstruir string
                chunk = " ".join(chunk_words)
                
                if len(chunk_words) < 15: # Descartar restos muy pequeños al final
                    continue
                
                file_chunks.append(chunk)
                
        if file_chunks:
            chunks_pdfs[pdf_file] = file_chunks
            print(f"Procesado {pdf_file}: {len(file_chunks)} chunks.")
            
    except Exception as e:
        print(f"Error leyendo {pdf_file}: {e}")

# Verificación
all_chunks = []
for k, v in chunks_pdfs.items():
    all_chunks.extend(v)

print(f"\nTotal chunks limpios: {len(all_chunks)}")
if len(all_chunks) > 0:
    print("Ejemplo de chunk limpio:")
    print(f"'{all_chunks[5]}'")

100%|██████████| 544/544 [00:20<00:00, 26.51it/s]


Procesado /home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/El Imperio Final.pdf: 1055 chunks.


100%|██████████| 607/607 [00:23<00:00, 26.02it/s]


Procesado /home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/El Heroe de las Eras. (Ed. revisada).pdf: 1167 chunks.


100%|██████████| 639/639 [00:23<00:00, 27.13it/s]

Procesado /home/alumno.upv.es/dnargar/tlh_trabajo/pdfs/El Pozo de la Ascension. (Ed. revisada).pdf: 1216 chunks.

Total chunks limpios: 3438
Ejemplo de chunk limpio:
'son capaces de dominarlos todos ahora kelsier el superviviente el único que ha logrado huir de los pozos de hathsin ha encontrado a vin una pobre chica skaa con mucha suerte tal vez los dos unidos a la rebelión que los skaa intentan desde hace mil años puedan cambiar el mundo y la atroz dominación del lord legislador tras el sorprendente éxito mundial de elantris brandon sanderson ha logrado repetir la hazaña la trilogía nacidos de la bruma mistborn es una excepcional muestra de su nueva fantasía y el primer volumen el imperio final con una conclusión cerrada y grandiosa que permite su lectura como una novela única e independiente nos introduce gratamente en el mundo de los poderes alománticos de la bruma y la ceniza de la dominación y la rebelión con personajes inolvidables como el atormentado keisler en su papel de pigm

Funciones para obtener todos los chunks, la función para realizar la comparación entre embeddings y para obtener los chunks más importantes

In [ ]:
def get_all_chunks(chunks_dict):
    all_chunks = []
    for chunks in chunks_dict.values():
        all_chunks.extend(chunks)
    return all_chunks

def cosine_similarity(v1, v2):
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0: return 0.0
    return np.dot(v1, v2) / (norm_v1 * norm_v2)

def get_relevant_chunks(query, encoder, tokenizer, chunks, seq_len, top_k=5):
    query_emb = get_page_embeddings(encoder, tokenizer, [query], seq_len=seq_len)[0]
    
    similarities = []
    for i, chunk in enumerate(chunks):
        chunk_emb = get_page_embeddings(encoder, tokenizer, [chunk], seq_len=seq_len)[0]
        sim = cosine_similarity(query_emb, chunk_emb)
        similarities.append((sim, chunk))
    
    similarities.sort(reverse=True, key=lambda x: x[0])
    return similarities[:top_k]

## Entrenamos el modelo

In [ ]:
d_model = 128

def get_all_chunks(chunks_dict):
    all_chunks = []
    for chunks in chunks_dict.values():
        all_chunks.extend(chunks)
    return all_chunks

all_chunks_pdfs = get_all_chunks(chunks_pdfs)

# Entrenar

encoder_entrenado, tokenizer = train_model(all_chunks_pdfs, d_model=d_model, seq_len=seq_len, epochs=50)

question = "la guerra contra los skaa"
print(f"\nPregunta: {question}")
print(tokenizer.encode(question, seq_len))

relevant_chunks = get_relevant_chunks(question, encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len=seq_len, top_k=3)

embedding_question = get_page_embeddings(encoder_entrenado, tokenizer, [question], seq_len=seq_len)[0]

for score, chunk in relevant_chunks:
    print(f"Similitud: {score:.4f}, Chunk: {chunk}")

Vocab Size: 27397


 10%|█         | 5/50 [00:25<03:47,  5.06s/it]

Epoch 5, Loss: 6.8900


 20%|██        | 10/50 [00:50<03:21,  5.04s/it]

Epoch 10, Loss: 6.7729


...

 90%|█████████ | 45/50 [03:50<00:28,  5.69s/it]

Epoch 45, Loss: 6.3487


100%|██████████| 50/50 [04:15<00:00,  5.11s/it]

Epoch 50, Loss: 6.2922

Pregunta: la guerra contra los skaa
tensor([    3, 15775, 13459,  6416, 16455, 24295,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
    

Similitud: 0.9841, Chunk: gente que se internaba en ellas brumoso alomántico que puede quemar solo un metal son mucho más corrientes que los nacidos de la bruma nota en la alomancia el alomántico tiene un poder o todos ellos no hay intermedios con dos o tres el lord legislador y sus sacerdotes siempre enseñaron que había solo ocho tipos de brumosos basados en los ocho primeros metales alománticos buscador brumoso que puede quemar bronce camon el antiguo jefe de la banda de vin un hombre duro que a menudo la golpeaba fue expulsado por kelsier los inquisidores acabaron matándolo cantón división del ministerio del acero
Similitud: 0.9802, Chunk: ahora es oficial del ejército de elend guardador terris a menudo se emplea el término guardador para referirse a los feruquimistas los guardadores son en realidad una organización de feruquimistas dedicados a descubrir y luego memorizar todo el conocimiento y las religiones que existían antes de la ascensión el lord legislador les dio caza hasta c

In [17]:
question = "el secreto del lord legislador"
print(f"\nPregunta: {question}")
print(tokenizer.encode(question, seq_len))

relevant_chunks = get_relevant_chunks(question, encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len=seq_len, top_k=3)

embedding_question = get_page_embeddings(encoder_entrenado, tokenizer, [question], seq_len=seq_len)[0]

for score, chunk in relevant_chunks:
    print(f"Similitud: {score:.4f}, Chunk: {chunk}")


Pregunta: el secreto del lord legislador
tensor([    3,  9812, 23776,  7753, 16453, 15954,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0, 

In [18]:
words = ["skaa", "lord", "legislador", "guerra", "vin", "kelsier", "sazed"]

similitudes = []

for word in words:
    embeddings = encoder_entrenado.embedding.embedding.weight.data.cpu().numpy()
    embedding_word = embeddings[tokenizer.word2idx.get(word, tokenizer.UNK_TOKEN)]
    for word_2 in words:
        embedding_word_2 = embeddings[tokenizer.word2idx.get(word_2, tokenizer.UNK_TOKEN)]
        sim = cosine_similarity(embedding_word, embedding_word_2)
        #print(f"  Similitud de {word} con {word_2}: {sim:.4f}")
        similitudes.append((sim, word, word_2))

for sim, w1, w2 in sorted(similitudes, reverse=True):
    if w1 != w2:
        print(f"Similitud de {w1} con {w2}: {sim:.4f}")

Similitud de lord con legislador: 0.2391
Similitud de legislador con lord: 0.2391
Similitud de legislador con kelsier: 0.2177
Similitud de kelsier con legislador: 0.2177
Similitud de sazed con lord: 0.1970
Similitud de lord con sazed: 0.1970
Similitud de skaa con kelsier: 0.1585
Similitud de kelsier con skaa: 0.1585
Similitud de lord con kelsier: 0.1466
Similitud de kelsier con lord: 0.1466
Similitud de sazed con kelsier: 0.1329
Similitud de kelsier con sazed: 0.1329
Similitud de vin con skaa: 0.1128
Similitud de skaa con vin: 0.1128
Similitud de skaa con lord: 0.0903
Similitud de lord con skaa: 0.0903
Similitud de skaa con legislador: 0.0887
Similitud de legislador con skaa: 0.0887
Similitud de vin con lord: 0.0731
Similitud de lord con vin: 0.0731
Similitud de sazed con guerra: 0.0467
Similitud de guerra con sazed: 0.0467
Similitud de skaa con sazed: 0.0425
Similitud de sazed con skaa: 0.0425
Similitud de vin con guerra: 0.0162
Similitud de guerra con vin: 0.0162
Similitud de sazed c

In [ ]:
import random
import torch.nn.functional as F

def evaluar_modelo(encoder, tokenizer, paginas, seq_len, k=5, num_muestras=50, ratio_query=0.5):
    """
    ratio_query: 0.5 significa que usamos la mitad del texto como consulta.
                 0.1 significa que usamos el 10% del texto.
    """
    print(f"--- Evaluación (Ratio: {ratio_query}, Top-K: {k}) ---")
    
    lista_numpy = get_page_embeddings(encoder, tokenizer, paginas, seq_len)
    db_embeddings = torch.tensor(np.array(lista_numpy))
    
    aciertos = 0
    indices_prueba = list(range(len(paginas)))
    random.shuffle(indices_prueba)
    indices_prueba = indices_prueba[:num_muestras]
    
    for idx_real in indices_prueba:
        texto_chunk = paginas[idx_real]
        palabras = texto_chunk.split()
        
        if len(palabras) < 10: continue

        # Calculamos longitud basada en el porcentaje
        target_len = int(len(palabras) * ratio_query)
        
        # Controlamos los límites de la longitud de la query
        len_query = max(target_len, 5)
        len_query = min(len_query, seq_len - 2)
        
        # Si el chunk es mas corto que seq_len, limitamos
        limit_start = len(palabras) - len_query
        if limit_start <= 0: 
            start_idx = 0
            len_query = len(palabras) # Usar todo si es muy corto
        else:
            start_idx = random.randint(0, limit_start)
            
        palabras_query = palabras[start_idx : start_idx + len_query]
        query_text = " ".join(palabras_query)
        
        vec_query_np = get_page_embeddings(encoder, tokenizer, [query_text], seq_len)[0]
        vec_query = torch.tensor(vec_query_np).unsqueeze(0)
        
        similitudes = F.cosine_similarity(vec_query, db_embeddings)
        vals, top_indices = torch.topk(similitudes, k=k)
        
        if idx_real in top_indices.tolist():
            aciertos += 1

    accuracy = (aciertos / len(indices_prueba)) * 100
    print(f"Ratio {ratio_query} -> Precisión: {accuracy:.2f}%")
    return accuracy

# Pruebas top-k y el token CLS (K=1 K=3, K=5)

In [ ]:
import os

if os.path.exists("encoder_mlm_cls.pth"):
    print("Cargando modelo entrenado desde disco...")
    d_model = 128
    seq_len = 256
    all_chunks_pdfs = get_all_chunks(chunks_pdfs)
    full_text = " ".join(all_chunks_pdfs)
    tokenizer = SimpleTokenizer(full_text)
    base_encoder = Encoder(d_model=d_model, vocab_size=tokenizer.vocab_size, 
                           seq_len=seq_len, nb_layers=6, n_head=8, d_head=d_model//8)
    encoder_entrenado = EncoderForMaskedLM(base_encoder, d_model, tokenizer.vocab_size)
    encoder_entrenado.load_state_dict(torch.load("encoder_mlm_cls.pth", map_location=device))
    encoder_entrenado.to(device)
    encoder_entrenado.eval()
    encoder_entrenado = encoder_entrenado.encoder

    print("EXPERIMENTOS K=1")

    print("EXPERIMENTO 1: Query Larga (50% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.5)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.3)
                        for _ in range(3)]) / 3

    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.1)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("EXPERIMENTOS K=3")

    print("EXPERIMENTO 1: Query Larga (50% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.5)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.3)
                        for _ in range(3)]) / 3

    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.1)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")


    print("EXPERIMENTOS K=5")
    print("EXPERIMENTO 1: Query Larga (50% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.5)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.3)
                        for _ in range(3)]) / 3

    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

    print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
    acc_media = np.sum([evaluar_modelo(encoder_entrenado, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.1)
                        for _ in range(3)]) / 3
    print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

Cargando modelo entrenado desde disco...
EXPERIMENTOS K=1
EXPERIMENTO 1: Query Larga (50% del texto original)
--- Evaluación (Ratio: 0.5, Top-K: 1) ---
Ratio 0.5 -> Precisión: 10.00%
--- Evaluación (Ratio: 0.5, Top-K: 1) ---
Ratio 0.5 -> Precisión: 14.00%
--- Evaluación (Ratio: 0.5, Top-K: 1) ---
Ratio 0.5 -> Precisión: 20.00%
Precisión promedio (3 ejecuciones): 14.67%

EXPERIMENTO 2: Query Media (30% del texto original)
--- Evaluación (Ratio: 0.3, Top-K: 1) ---
Ratio 0.3 -> Precisión: 2.00%
--- Evaluación (Ratio: 0.3, Top-K: 1) ---
Ratio 0.3 -> Precisión: 6.00%
--- Evaluación (Ratio: 0.3, Top-K: 1) ---
Ratio 0.3 -> Precisión: 4.00%
Precisión promedio (3 ejecuciones): 4.00%

EXPERIMENTO 3: Query Corta (10% del texto original - Difícil)
--- Evaluación (Ratio: 0.1, Top-K: 1) ---
Ratio 0.1 -> Precisión: 2.00%
--- Evaluación (Ratio: 0.1, Top-K: 1) ---
Ratio 0.1 -> Precisión: 0.00%
--- Evaluación (Ratio: 0.1, Top-K: 1) ---
Ratio 0.1 -> Precisión: 0.00%
Precisión promedio (3 ejecuciones): 0.